# UAE Mobile Intelligence - WorldPop Population Collection

Downloads the WorldPop R2025A constrained population raster for the UAE
(2026, 100m resolution) and validates it against the national total (~11.5M)
cited in the challenge brief.

Source: https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2026/ARE/v1/100m/constrained/are_pop_2026_CN_100m_R2025A_v1.tif
Licence: CC BY 4.0

In [1]:
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import rasterio

In [2]:
WORLDPOP_URL = (
    "https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/"
    "2026/ARE/v1/100m/constrained/are_pop_2026_CN_100m_R2025A_v1.tif"
)

RAW_DIR = Path("../data/raw/worldpop")
RAW_DIR.mkdir(parents=True, exist_ok=True)

raster_path = RAW_DIR / "are_pop_2026_CN_100m_R2025A_v1.tif"

In [3]:
# Download (skip if already present)

if raster_path.exists():
    print("Already downloaded:", raster_path)
else:
    print("Downloading:", WORLDPOP_URL)
    urllib.request.urlretrieve(WORLDPOP_URL, raster_path)
    print("Saved to:", raster_path)

size_mb = raster_path.stat().st_size / (1024 ** 2)
print(f"File size: {size_mb:.2f} MB")

Downloading: https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2026/ARE/v1/100m/constrained/are_pop_2026_CN_100m_R2025A_v1.tif


Saved to: ..\data\raw\worldpop\are_pop_2026_CN_100m_R2025A_v1.tif
File size: 6.26 MB


In [4]:
# Checksum, for reproducibility record-keeping

sha256 = hashlib.sha256(raster_path.read_bytes()).hexdigest()
print("SHA256:", sha256)

SHA256: 4d07f6103987d192adedb790cee884296d927221a2a8384a3f0ae62d6d8e44ba


In [5]:
# --------------------------------------------
# Inspect raster metadata
# --------------------------------------------

with rasterio.open(raster_path) as src:
    print("CRS:", src.crs)
    print("Shape (rows, cols):", src.shape)
    print("Resolution (deg):", src.res)
    print("Bounds:", src.bounds)
    print("Nodata:", src.nodata)
    print("Dtype:", src.dtypes)

CRS: EPSG:4326
Shape (rows, cols): (4124, 5861)
Resolution (deg): (0.00083333333, 0.00083333333)
Bounds: BoundingBox(left=51.498332407339994, bottom=22.632500245469984, right=56.382499054469996, top=26.069166898389984)
Nodata: -99999.0
Dtype: ('float32',)


In [6]:
# --------------------------------------------
# Validate national total against brief (~11.5M)
# --------------------------------------------

with rasterio.open(raster_path) as src:
    band = src.read(1, masked=True)

national_total = float(band.sum())
populated_cells = int((band.filled(0) > 0).sum())
total_cells = band.size

print(f"National population total: {national_total:,.0f}")
print(f"Populated cells: {populated_cells:,} / {total_cells:,} "
      f"({populated_cells / total_cells:.1%})")

National population total: 11,476,873
Populated cells: 1,127,338 / 24,170,764 (4.7%)


## Notes

- Product: R2025A constrained, 2026, 100m — matches the exact file named in
  the challenge brief.
- "Constrained" means population is placed only where built settlement
  exists (per the WorldPop release notes), which suits tile-level exposure
  weighting later in the pipeline.
- Years after 2020 in this series (including 2026) are modelled projections,
  not census counts, and R2025A is marked an alpha product. Treat totals as
  estimates, not ground truth.
- No cross-year population weighting is needed since this raster already
  matches the 2026 Ookla measurement window.
- Aggregation into H3 zones (population-conserving) happens in a later
  processing notebook, once the geographic unit is finalized.